In [1]:
from pandas import DataFrame

In [2]:
from streamlit_apps.apps.streamlit_app_research.application.services.asset_screening import AssetScreening10yPrice10yITRService, get_eligible_assets_10yPrice10yITR
df_1: DataFrame = get_eligible_assets_10yPrice10yITR(service=AssetScreening10yPrice10yITRService())

2026-09-18 12:37:53.718 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-18 12:37:53.719 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-18 12:37:53.723 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-18 12:37:53.725 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-18 12:37:53.726 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-18 12:37:53.727 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-18 12:37:53.728 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-18 12:37:53.729 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-

In [3]:
df_1.head(5)

,cod,asset,codeCVM,anos_diferenca_preco,anos_diferenca_itr,ma_volume_financeiro
0,PETR4,PETROBRAS,9512,26,15,4.881156e+08
1,BRAP4,BRADESPAR,18724,26,12,2.565020e+08
2,PETR3,PETROBRAS,9512,26,15,7.098492e+07
3,ITUB4,ITAUUNIBANCO,19348,25,15,4.709112e+07
4,B3SA3,B3,21610,18,15,3.337397e+07


In [4]:
from data_providers.providers.yfinance_price_provider import YFinancePriceProvider
from streamlit_apps.apps.streamlit_app_research.application.services.asset_demonstration_service import AssetDemonstrationService

asset_demonstration_service = AssetDemonstrationService()

def _get_ret_stats(df: DataFrame) -> DataFrame:
    
    ret_stats = {}

    for row in df.itertuples():
        price_df = YFinancePriceProvider().get_asset_price(
            tickers=row.cod + ".SA", period="10y"
        )
        
        ret = price_df["Adj Close"].pct_change(1)
        
        try:
            divida_liquida = asset_demonstration_service.get_divida_liquida(row.codeCVM)["VL_CONTA_TRI"]
            media_divida_liquida = divida_liquida.mean()
            std_divida_liquida = divida_liquida.std()
            mediana_divida_liquida = divida_liquida.median()
        except:
            media_divida_liquida = None
            std_divida_liquida = None
            mediana_divida_liquida = None
            
        try:
            lucro_liquido = asset_demonstration_service.get_lucro_liquido(row.codeCVM)["VL_CONTA_TRI"]
            media_lucro_liquido = lucro_liquido.mean()
            std_lucro_liquido = lucro_liquido.std()
            mediana_lucro_liquido = lucro_liquido.median()
        except:
            media_lucro_liquido = None
            std_lucro_liquido = None
            mediana_lucro_liquido = None
            
        try:
            patrimonio_liquido = asset_demonstration_service.get_patrimonio_liquido(row.codeCVM)["VL_CONTA_TRI"]
            media_patrimonio_liquido = patrimonio_liquido.mean()
            std_patrimonio_liquido = patrimonio_liquido.std()
            mediana_patrimonio_liquido = patrimonio_liquido.median()
        except:
            media_patrimonio_liquido = None
            std_patrimonio_liquido = None
            mediana_patrimonio_liquido = None
        
        try:
            media_indice_de_alavancagem_financeira = (divida_liquida / patrimonio_liquido).mean()
            std_indice_de_alavancagem_financeira = (divida_liquida / patrimonio_liquido).std()
            mediana_indice_de_alavancagem_financeira = (divida_liquida / patrimonio_liquido).median()
        except:
            media_indice_de_alavancagem_financeira = None
            std_indice_de_alavancagem_financeira = None
            mediana_indice_de_alavancagem_financeira = None
                
        ret_stats[row.cod] = {
            "last_price": price_df["Close"].iloc[-1],
            "std_volume_financeiro": price_df["Volume"].std(),
            "media_ret": ret.mean(),
            "std_ret": ret.std(),
            "mediana_ret": ret.median(),
            "media_divida_liquida": media_divida_liquida,
            "std_divida_liquida": std_divida_liquida,
            "mediana_divida_liquida": mediana_divida_liquida,
            "media_lucro_liquido": media_lucro_liquido,
            "std_lucro_liquido": std_lucro_liquido,
            "mediana_lucro_liquido": mediana_lucro_liquido,
            "media_patrimonio_liquido": media_patrimonio_liquido,
            "std_patrimonio_liquido": std_patrimonio_liquido,
            "mediana_patrimonio_liquido": mediana_patrimonio_liquido,
            "media_indice_de_alavancagem_financeira": media_indice_de_alavancagem_financeira,
            "std_indice_de_alavancagem_financeira": std_indice_de_alavancagem_financeira,
            "mediana_indice_de_alavancagem_financeira": mediana_indice_de_alavancagem_financeira
            
        }
        
    return (
        DataFrame.from_dict(ret_stats, orient="index")
        .rename_axis("cod")
        .reset_index()
    )

df_2 = df_1.merge(_get_ret_stats(df_1), on="cod", how="left")

2026-09-18 12:38:47.082 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-18 12:38:47.083 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-18 12:38:47.084 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-18 12:38:47.085 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-18 12:38:47.109 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-18 12:38:47.131 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-18 12:38:47.150 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-18 12:38:47.151 WARNING streamlit.runtime.scriptrunner_ut

     CD_CONTA                                           DS_CONTA
61          2                                      Passivo Total
2880     2.01  Passivos Financeiros ao Valor Justo através do...
59       2.01               Passivos Financeiros para Negociação
4069  2.01.01                                        Derivativos
4067  2.01.02                                 Notas Estruturadas
...       ...                                                ...
32    2.08.05                        Lucros/Prejuízos Acumulados
29    2.08.06                   Ajustes de Avaliação Patrimonial
27    2.08.07                    Ajustes Acumulados de Conversão
25    2.08.08                      Outros Resultados Abrangentes
23    2.08.09      Participação dos Acionistas Não Controladores

[91 rows x 2 columns]
        CD_CONTA                                           DS_CONTA
31          3.01               Receitas da Intermediação Financeira
3602     3.01.01  Rec de Juros e Rend de Ativos Fin ao Custo 

2026-09-18 12:38:50.150 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-18 12:38:50.234 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-18 12:38:50.268 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-18 12:38:50.838 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-18 12:38:50.919 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-18 12:38:50.959 WARNING streamlit.run

        CD_CONTA                                           DS_CONTA
78             2                                      Passivo Total
575      2,06.01                  Derivativos utilizados como hedge
577      2,06.02                   Passivos por contratos de seguro
579      2,06.03                                  Outras obrigações
4202        2.01  Passivos Financeiros Avaliados ao Valor Justo ...
...          ...                                                ...
1676  2.08.08.04  Ganhos e Perdas - Hedge de Fluxo de Caixa e de...
1843  2.08.08.04         Hegde de Fleuxo de Caixa e de Investimento
1939  2.08.08.04          Hegde de Fluxo de Caixa e de Investimento
3467  2.08.08.05  Ativos Fin Mensurados ao Valor Justo por Meio ...
28       2.08.09      Participação dos Acionistas Não Controladores

[175 rows x 2 columns]


2026-09-18 12:39:16.184 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-18 12:39:16.240 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-18 12:39:16.259 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-18 12:39:16.820 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-18 12:39:16.890 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-18 12:39:16.919 WARNING streamlit.run

In [5]:
df_3 = df_2.dropna()
df_3.head(5)

,cod,asset,codeCVM,anos_diferenca_preco,anos_diferenca_itr,ma_volume_financeiro,last_price,std_volume_financeiro,media_ret,std_ret,...,mediana_divida_liquida,media_lucro_liquido,std_lucro_liquido,mediana_lucro_liquido,media_patrimonio_liquido,std_patrimonio_liquido,mediana_patrimonio_liquido,media_indice_de_alavancagem_financeira,std_indice_de_alavancagem_financeira,mediana_indice_de_alavancagem_financeira
0,PETR4,PETROBRAS,9512,26,15,4.881156e+08,48.619999,3.355914e+07,0.001422,0.025677,...,285985000.0,1.186342e+07,2.109088e+07,6427298.5,3.324178e+08,5.758449e+07,335846842.5,0.867373,0.345893,0.801463
1,BRAP4,BRADESPAR,18724,26,12,2.565020e+08,21.059999,3.191606e+06,0.001410,0.022861,...,593318.0,3.630645e+05,7.825377e+05,329133.0,9.164912e+06,1.498343e+06,9000800.0,0.029821,0.083887,0.067143
2,PETR3,PETROBRAS,9512,26,15,7.098492e+07,53.910000,1.040316e+07,0.001386,0.025889,...,285985000.0,1.186342e+07,2.109088e+07,6427298.5,3.324178e+08,5.758449e+07,335846842.5,0.867373,0.345893,0.801463
4,B3SA3,B3,21610,18,15,3.337397e+07,17.540001,2.073248e+07,0.000868,0.023431,...,5338738.0,7.015838e+05,4.790457e+05,687078.0,2.101333e+07,2.553007e+06,19704926.5,0.272178,0.232067,0.217756
5,COGN3,COGNA ON,17973,14,15,2.525699e+07,2.230000,2.530717e+07,-0.000128,0.032904,...,4191268.0,4.890026e+04,6.196454e+05,96880.0,1.100923e+07,5.119435e+06,12512566.0,0.333565,0.268080,0.366089


In [6]:
df_3.shape

(56, 23)

In [7]:
df_4 = df_3[df_3["media_lucro_liquido"] > 0].copy()
df_4 = df_4[df_4["last_price"] > 9]

In [8]:
df_4.shape

(45, 23)

In [9]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

features = [
    # "ma_volume_financeiro",
    "std_volume_financeiro",
    # "media_ret",
    "std_ret",
    # "media_divida_liquida",
    # "std_divida_liquida",
    "mediana_divida_liquida",
    # "media_lucro_liquido",
    # "std_lucro_liquido",
    # "mediana_lucro_liquido",
    # "media_patrimonio_liquido",
    # "std_patrimonio_liquido",
    # "mediana_patrimonio_liquido",
    # "media_indice_de_alavancagem_financeira",
    # "std_indice_de_alavancagem_financeira",
    # "mediana_indice_de_alavancagem_financeira"
]

X = df_4[features].copy()

# log1p nas colunas com cauda pesada (volume e lucro têm ordens de magnitude de diferença)
# lucro pode ser negativo em alguns ativos -> usar signo * log1p(abs()) preserva o sinal
for col in [
    # "ma_volume_financeiro",
    "std_volume_financeiro",
    # "media_ret",
    # "std_ret",
    # "media_divida_liquida",
    # "std_divida_liquida",
    "mediana_divida_liquida",
    # "media_lucro_liquido",
    # "std_lucro_liquido",
    # "mediana_lucro_liquido",
    # "media_patrimonio_liquido",
    # "std_patrimonio_liquido",
    # "mediana_patrimonio_liquido",
    # "media_indice_de_alavancagem_financeira",
    # "std_indice_de_alavancagem_financeira",
    # "mediana_indice_de_alavancagem_financeira"
    
]:
    X[col] = np.sign(X[col]) * np.log1p(np.abs(X[col]))

X_scaled = StandardScaler().fit_transform(X)

# escolher k via silhouette (com 53 linhas, testar poucos k é suficiente)
scores = {}
k = 2
for k in range(k, 8):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)

best_k = max(scores, key=scores.get)
print("silhouette por k:", scores)
print("melhor k:", best_k)

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_4["cluster"] = kmeans.fit_predict(X_scaled)


silhouette por k: {2: 0.3318199477891246, 3: 0.406078295328368, 4: 0.42593476842806277, 5: 0.3838366005143143, 6: 0.37430781763289256, 7: 0.3430659196477}
melhor k: 4


In [10]:
df_4.groupby("cluster")[features].mean()

,std_volume_financeiro,std_ret,mediana_divida_liquida
cluster,,,
0,5.430071e+06,0.030507,9.442989e+06
1,1.232437e+07,0.023071,5.517528e+07
2,4.703662e+06,0.018887,-7.611571e+06
3,2.604712e+06,0.019246,5.798833e+06


In [11]:
df_4["cluster"].value_counts()

cluster
3    18
1    13
0     9
2     5
Name: count, dtype: int64

In [12]:
df_4.tail(3)

,cod,asset,codeCVM,anos_diferenca_preco,anos_diferenca_itr,ma_volume_financeiro,last_price,std_volume_financeiro,media_ret,std_ret,...,media_lucro_liquido,std_lucro_liquido,mediana_lucro_liquido,media_patrimonio_liquido,std_patrimonio_liquido,mediana_patrimonio_liquido,media_indice_de_alavancagem_financeira,std_indice_de_alavancagem_financeira,mediana_indice_de_alavancagem_financeira,cluster
54,ISAE4,ISA ENERGIA,18376,26,15,1.137569e+06,27.209999,1.607580e+06,0.000633,0.013829,...,5.199337e+05,622267.470955,454304.0,1.143528e+07,5.684068e+06,11267653.0,0.380676,0.186063,0.375380,3
55,VIVT3,TELEF BRASIL,17671,26,15,1.124827e+06,30.799999,3.086310e+06,0.000609,0.016561,...,1.260763e+06,512266.133627,1213156.5,6.173527e+07,1.311969e+07,68693209.5,0.100171,0.062728,0.071628,3
56,AZZA3,AZZAS 2154,22349,15,15,8.595674e+05,14.500000,1.352001e+06,0.000237,0.026663,...,5.775522e+04,77597.883619,33601.0,1.943669e+06,2.425235e+06,693658.0,0.226634,0.157189,0.185030,3


In [13]:
# Interpretação dos clusters:
#
# Cluster 0:
# - Variabilidade do volume financeiro intermediária (≈ R$ 5,43 milhões).
# - Maior volatilidade dos retornos (≈ 3,05%).
# - Dívida líquida mediana positiva (≈ R$ 9,08 milhões).
# - Perfil de ativos mais volátil, com endividamento líquido moderado.
#
# Cluster 1:
# - Maior variabilidade do volume financeiro (≈ R$ 12,32 milhões).
# - Volatilidade dos retornos intermediária (≈ 2,31%).
# - Maior dívida líquida mediana (≈ R$ 54,43 milhões).
# - Perfil associado a maior escala financeira e maior endividamento líquido.
#
# Cluster 2:
# - Variabilidade do volume financeiro relativamente alta (≈ R$ 4,70 milhões).
# - Menor volatilidade dos retornos (≈ 1,89%).
# - Dívida líquida mediana negativa (≈ -R$ 7,57 milhões).
# - Perfil mais estável e com posição de caixa superior à dívida na mediana.
#
# Cluster 3:
# - Menor variabilidade do volume financeiro (≈ R$ 2,60 milhões).
# - Baixa volatilidade dos retornos (≈ 1,93%).
# - Dívida líquida mediana positiva, porém baixa (≈ R$ 4,82 milhões).
# - Perfil de menor atividade financeira e comportamento relativamente estável.
#
# Síntese:
# - Cluster 1: maior volume e maior endividamento.
# - Cluster 0: maior volatilidade dos retornos.
# - Cluster 2: baixa volatilidade e dívida líquida negativa.
# - Cluster 3: menor variabilidade do volume e baixa volatilidade.
#
# Observação:
# - Como volume financeiro e dívida líquida são medidas absolutas, parte da separação
#   pode estar relacionada ao porte das empresas.
# - Para uma interpretação econômica mais robusta, seria interessante utilizar também
#   métricas relativas, como dívida líquida/EBITDA, dívida líquida/patrimônio líquido
#   e margem líquida.

In [14]:
import plotly.express as px
from sklearn.decomposition import PCA

# TODO: Atualizar o número de componentes conforme a quantidade de features utilizadas no estudo.

pca = PCA(n_components=2, random_state=42)
components = pca.fit_transform(X_scaled)

df_4["pca_1"] = components[:, 0]
df_4["pca_2"] = components[:, 1]

ativo_destaque = "KLBN11"
ativo = df_4[df_4["cod"] == ativo_destaque]

var_explained = pca.explained_variance_ratio_

fig = px.scatter(
    df_4,
    x="pca_1",
    y="pca_2",
    color=df_4["cluster"].astype(str),
    hover_name="cod",
    hover_data={
        "cod": True,
        "asset": True,
        "media_ret": ":.4f",
        "std_ret": ":.4f",
        "ma_volume_financeiro": ":.2e",
        "std_volume_financeiro": ":.2e",
        "media_lucro_liquido": ":.2e",
        "std_lucro_liquido": ":.2e",
        "media_divida_liquida": ":.2e",
        "std_divida_liquida": ":.2e",
        
        "pca_1": False,
        "pca_2": False,
    },
    labels={
        "pca_1": f"PC1 ({var_explained[0]:.1%} var.)",
        "pca_2": f"PC2 ({var_explained[1]:.1%} var.)",
        "color": "Cluster",
    },
    title="Clusters de ativos — projeção PCA 2D",
    template="plotly_white",
    color_discrete_sequence=px.colors.qualitative.Set2,
)

fig.add_scatter(
    x=ativo["pca_1"],
    y=ativo["pca_2"],
    mode="markers+text",
    text=ativo["cod"],
    textposition="top center",
    marker=dict(
        size=14,
        color="red",
        symbol="diamond",
        line=dict(width=1, color="black"),
    ),
    name=f"Ativo: {ativo_destaque}",
)

fig.update_traces(
    marker=dict(size=12, line=dict(width=1, color="white")),
    selector=dict(mode="markers"),
)

fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)
fig.add_vline(x=0, line_dash="dash", line_color="gray", line_width=1)


fig.update_layout(
    legend_title_text="Cluster",
    hoverlabel=dict(bgcolor="white", font_size=13),
    height=550,
)

fig.show()

In [15]:
df_4.tail(3)

,cod,asset,codeCVM,anos_diferenca_preco,anos_diferenca_itr,ma_volume_financeiro,last_price,std_volume_financeiro,media_ret,std_ret,...,mediana_lucro_liquido,media_patrimonio_liquido,std_patrimonio_liquido,mediana_patrimonio_liquido,media_indice_de_alavancagem_financeira,std_indice_de_alavancagem_financeira,mediana_indice_de_alavancagem_financeira,cluster,pca_1,pca_2
54,ISAE4,ISA ENERGIA,18376,26,15,1.137569e+06,27.209999,1.607580e+06,0.000633,0.013829,...,454304.0,1.143528e+07,5.684068e+06,11267653.0,0.380676,0.186063,0.375380,3,-1.636388,1.244547
55,VIVT3,TELEF BRASIL,17671,26,15,1.124827e+06,30.799999,3.086310e+06,0.000609,0.016561,...,1213156.5,6.173527e+07,1.311969e+07,68693209.5,0.100171,0.062728,0.071628,3,-0.801376,0.796814
56,AZZA3,AZZAS 2154,22349,15,15,8.595674e+05,14.500000,1.352001e+06,0.000237,0.026663,...,33601.0,1.943669e+06,2.425235e+06,693658.0,0.226634,0.157189,0.185030,3,-0.415298,0.434576


In [16]:
# Avalia diferentes valores de k no KMeans para identificar uma quantidade
# adequada de clusters, considerando a separação dos grupos (silhouette)
# e a distribuição dos ativos entre eles.

import pandas as pd

for k in range(2, 8):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_scaled)

    silhouette = silhouette_score(X_scaled, labels)

    sizes = pd.Series(labels).value_counts().sort_index()

    print(
        f"k={k} | "
        f"silhouette={silhouette:.3f} | "
        f"tamanho={sizes.tolist()}"
    )

k=2 | silhouette=0.332 | tamanho=[23, 22]
k=3 | silhouette=0.406 | tamanho=[22, 5, 18]
k=4 | silhouette=0.426 | tamanho=[9, 13, 5, 18]
k=5 | silhouette=0.384 | tamanho=[13, 8, 5, 8, 11]
k=6 | silhouette=0.374 | tamanho=[8, 8, 4, 11, 1, 13]
k=7 | silhouette=0.343 | tamanho=[7, 8, 4, 12, 1, 8, 5]
